Monthly data: Natural Gas prices, consumer price energy, production in industry

In [ ]:
import pandas as pd


def monthly_upsampled(file_path1, column_name1, name_file):
  # Load your CSV file
  file_path = file_path1  # Replace with your file path
  df = pd.read_csv(file_path, parse_dates=["date"], dayfirst=False)  # Uses mm/dd/yyyy format
  df.set_index("date", inplace=True)

  # print("Raw Data Head:")
  # print(df.head())


  # Detect low-frequency columns (e.g., monthly) manually or by checking NaNs
  low_freq_cols = [column_name1] # Replace with your columns
  daily_cols = [col for col in df.columns if col not in low_freq_cols]

  # Separate data
  daily_data = df[daily_cols].copy()
  monthly_data = df[low_freq_cols].dropna(how="all")  # Drops rows where all values are NaN

  monthly_upsampled = monthly_data.reindex(daily_data.index, method="ffill")  # Forward-fill

  monthly_upsampled = monthly_data.resample("D").interpolate("linear")  # Linear interpolation

  final_df = pd.concat([daily_data, monthly_upsampled], axis=1)

  # Optionally, backfill initial NaN values (e.g., if first month starts later)
  final_df[low_freq_cols] = final_df[low_freq_cols].bfill()

  print("\nFinal Dataset Head:")
  print(final_df.head())

  final_df.to_csv(name_file)






In [ ]:
#function to intrapolate from monthly to weekly
import pandas as pd

def weekly_upsampled(file_path1, column_name1, name_file):
    # Load your CSV file
    df = pd.read_csv(file_path1, parse_dates=["date"])
    df.set_index("date", inplace=True)

        # Ensure output folder exists
    os.makedirs("features_daily_weekly", exist_ok=True)

    # Full output path
    output_path = os.path.join("features_daily_weekly", name_file)

    # Identify which column(s) are lower frequency (e.g., monthly)
    low_freq_cols = [column_name1]
    high_freq_cols = [col for col in df.columns if col not in low_freq_cols]

    # Separate datasets
    high_freq_data = df[high_freq_cols].copy()
    low_freq_data = df[low_freq_cols].dropna(how="all")

    # Resample everything to weekly frequency (Sunday end of week by default)
    high_freq_weekly = high_freq_data.resample("W").mean()
    low_freq_weekly = low_freq_data.resample("W").interpolate("linear")

    # Merge datasets
    final_df = pd.concat([high_freq_weekly, low_freq_weekly], axis=1)

    # Optionally backfill missing values at the start
    final_df[low_freq_cols] = final_df[low_freq_cols].bfill()

    print("\nFinal Weekly Dataset Head:")
    print(final_df.head())

    # Save to CSV
    final_df.to_csv(output_path)



In [ ]:
#Intrapolating 5 year data from monthly to daily

#Natural Gas prices
monthly_upsampled("/content/Adv_ml_feature_engineering - natural gas prices.csv", "EU Natural Gas Prices", "natural_gas_daily.csv")

#consumer price energy index
monthly_upsampled("/content/Adv_ml_feature_engineering - consumer price index for energy.csv", "cpi", "cpi_daily.csv")

#production in industry
monthly_upsampled("/content/Adv_ml_feature_engineering - PRODUCTION IN INDUSTRY.csv", "euro-20","production_industry_daily.csv")

In [ ]:

#Intrapolating 10 year data from monthly to daily

#Natural Gas prices

monthly_upsampled("/content/10_year_dataset - Copy - natural_gas_prices_10.csv", "EU Natural Gas Prices", "10_natural_gas_daily.csv")

#consumer price energy index

monthly_upsampled("/content/10_year_dataset - Copy - consumer_price_index_10.csv", " Price Index for Consumer: Energy(electricity, heat, fuel) ", "10_cpi_daily.csv")

#production in industry

monthly_upsampled("/content/10_year_dataset - Copy - production_in_industry_10.csv","production volume in industry","10_production_industry_daily.csv")

In [ ]:
#Intrapolating 10 year data from monthly to weekly

#
weekly_upsampled("/content/10_year_dataset - Copy - natural_gas_prices_10.csv", "EU Natural Gas Prices", "10_natural_gas_weekly.csv")

#
weekly_upsampled("/content/10_year_dataset - Copy - consumer_price_index_10.csv", " Price Index for Consumer: Energy(electricity, heat, fuel) ", "10_cpi_weekly.csv")

  #
weekly_upsampled("/content/10_year_dataset - Copy - production_in_industry_10.csv","production volume in industry","10_production_industry_weekly.csv")



In [ ]:
#Intrapolating 5 year data from monthly to weekly




In [ ]:
import pandas as pd
import os

def daily_to_weekly(input_file, output_folder, filename, date_col="date", method="mean"):
    """
    Converts daily data to weekly data and saves it to a specified folder with a given filename.

    Parameters:
    - input_file: Path to the input CSV file with daily data.
    - output_folder: Folder where the output should be saved.
    - filename: Desired name for the output file (e.g. 'weekly_sentiment.csv').
    - date_col: Column containing the date values.
    - method: Aggregation method: 'mean', 'sum', or 'last'.
    """

    # Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Full output path
    output_path = os.path.join(output_folder, filename)

    # Load data
    df = pd.read_csv(input_file, parse_dates=[date_col])
    df.set_index(date_col, inplace=True)

    # Resample
    if method == "mean":
        df_weekly = df.resample("W").mean()
    elif method == "sum":
        df_weekly = df.resample("W").sum()
    elif method == "last":
        df_weekly = df.resample("W").last()
    else:
        raise ValueError("Unsupported method. Choose from 'mean', 'sum', or 'last'.")

    # Save
    df_weekly.to_csv(output_path)
    print(f"Weekly data saved to: {output_path}")
    return df_weekly


In [ ]:

daily_to_weekly("/content/10_year_dataset - Copy - coal_prices_10.csv","features_daily_weekly", "coal_daily_2_weekly.csv")

daily_to_weekly("/content/10_year_dataset - Copy - crude_oil_prices_10.csv","features_daily_weekly", "crude_daily_2_weekly.csv")

daily_to_weekly("/content/10_year_dataset - Copy - global_clean_index_10.csv","features_daily_weekly", "global_clean_daily_2_weekly.csv")

daily_to_weekly("/content/10_year_dataset - Copy - euro_stoxx_50_10.csv","features_daily_weekly", "euro_stoxx_daily_2_weekly.csv")

daily_to_weekly("/content/10_year_dataset - Copy - ecb_interest_rate_10.csv","features_daily_weekly", "ecb_interest_rate_daily_2_weekly.csv")

daily_to_weekly("/content/features/sentiment_matched.csv","features_daily_weekly", "sentiment_daily_2_weekly.csv")

GETTING ALL INTO 1 DATABASE

In [ ]:
import pandas as pd
import os

# Load datasets

def same_date (path_features, path_return):
  #df_labels = pd.read_csv("/content/Adv_ml_feature_engineering - Carbon Emissions Futures in EUR.csv")
  df_labels = pd.read_csv("/content/10_year_dataset - Copy - Carbon_Emissions_Futures_10.csv")
  df_features = pd.read_csv(path_features)

  # Convert first column to datetime
  df_labels.iloc[:, 0] = pd.to_datetime(df_labels.iloc[:, 0])
  df_features.iloc[:, 0] = pd.to_datetime(df_features.iloc[:, 0])

  # Sort and set index
  df_labels = df_labels.sort_values(by=df_labels.columns[0]).set_index(df_labels.columns[0])
  df_features = df_features.sort_values(by=df_features.columns[0]).set_index(df_features.columns[0])

  # Step 1: Ensure features cover all label dates
  full_dates = df_labels.index

  # Step 2: Reindex features to match label dates, fill forward
  df_features_aligned = df_features.reindex(full_dates, method='ffill')

  # Step 3: Drop label rows with no corresponding features (after reindex fill still NaN)
  valid_rows = df_features_aligned.notnull().all(axis=1)
  df_labels_final = df_labels[valid_rows]
  df_features_final = df_features_aligned[valid_rows]

  # Step 4: Reset index
  df_labels_final = df_labels_final.reset_index().rename(columns={'index': 'Date'})
  df_features_final = df_features_final.reset_index().rename(columns={'index': 'Date'})

  # Save results
  output_folder = "features2/"
  os.makedirs(output_folder, exist_ok=True)
  output_path = os.path.join(output_folder, path_return)
  # Save results
  df_labels_final.to_csv("labels_aligned.csv", index=False)
  df_features_final.to_csv(output_path, index=False)

  print("? Final features and labels aligned perfectly by date.")








In [ ]:
import pandas as pd

def merge_weekly_features_to_labels(feature_path, output_path):
    # Load datasets

    df_features = pd.read_csv(feature_path, parse_dates=["date"])
    df_labels = pd.read_csv("/content/10_year_dataset - Copy - Carbon_emissions_weekly_10.csv", parse_dates=["date"])

    os.makedirs("features_weekly", exist_ok=True)

    # Full output path
    output_path = os.path.join("features_weekly", output_path)

    # Step 1: Resample both to weekly frequency (ending Saturday to match label pattern)
    df_features_weekly = df_features.set_index("date").resample("W-SAT").mean().reset_index()
    df_labels_weekly = df_labels.set_index("date").resample("W-SAT").mean().reset_index()

    # Step 2: Left join ? keep all labels' weekly dates
    df_merged = pd.merge(df_labels_weekly, df_features_weekly, on="date", how="left")
    df_merged = df_merged.drop(columns=["label"])
    print("? Merged dataset:")
    print(df_merged.head())
    # Step 3: Save to CSV
    df_merged.to_csv(output_path, index=False)
    print(f"Merged dataset saved to: {output_path}")

    return df_merged


In [ ]:
#Getting same date for 5 year data, monthly intrapolated, daily frequency

#euro_stoxx
same_date ("/content/Adv_ml_feature_engineering - euro stoxx 50.csv","euro_stoxx_matched.csv")

#ecb interest rate
same_date ("/content/Adv_ml_feature_engineering - ECB iNTEREST RATE.csv","ecb_interest_rate_matched.csv")

#crude oil prices
same_date ("/content/Adv_ml_feature_engineering - CRUDE OIL PRICES.csv","crude_oil_prices_matched.csv")

#coal prices
same_date ("/content/Adv_ml_feature_engineering - COAL PRICES.csv","coal_prices_matched.csv")

#natural gas prices
same_date ("/content/Adv_ml_feature_engineering - natural gas prices.csv","natural_gas_prices_matched.csv")

#s&p 500
same_date ("/content/Adv_ml_feature_engineering - S&P 500.csv","s&p_500_matched.csv")

#consumer price
same_date ("/content/cpi_daily.csv","consumer_price_matched.csv")

#production in industry
same_date ("/content/production_industry_daily.csv","production_industry_matched.csv")


In [ ]:
#10 years monthly, daily frequency

#euro_stoxx
same_date ("/content/10_year_dataset - Copy - euro_stoxx_50_10.csv","10md_euro_stoxx_matched.csv")

#ecb interest rate
same_date ("/content/10_year_dataset - Copy - ecb_interest_rate_10.csv","10md_ecb_interest_rate_matched.csv")

#crude oil prices
same_date ("/content/10_year_dataset - Copy - crude_oil_prices_10.csv","10md_crude_oil_prices_matched.csv")

#coal prices
same_date ("/content/10_year_dataset - Copy - coal_prices_10.csv","10md_coal_prices_matched.csv")

#natural gas prices
same_date ("/content/10_natural_gas_daily.csv","10md_natural_gas_prices_matched.csv")

# #s&p 500
# same_date ("/content/Adv_ml_feature_engineering - S&P 500.csv","s&p_500_matched.csv")

#consumer price
same_date ("/content/10_cpi_daily.csv","10mdconsumer_price_matched.csv")

#production in industry
same_date ("/content/10_production_industry_daily.csv","10mdproduction_industry_matched.csv")

#global clean energy index
same_date ("/content/10_year_dataset - Copy - global_clean_index_10.csv","10md_global_clean_energy_index_matched.csv")


In [ ]:
# 10 years no monthly, daily frequency

#euro_stoxx
same_date ("/content/10_year_dataset - Copy - euro_stoxx_50_10.csv","10md_euro_stoxx_matched.csv")

#ecb interest rate
same_date ("/content/10_year_dataset - Copy - ecb_interest_rate_10.csv","10md_ecb_interest_rate_matched.csv")

#crude oil prices
same_date ("/content/10_year_dataset - Copy - crude_oil_prices_10.csv","10md_crude_oil_prices_matched.csv")

#coal prices
same_date ("/content/10_year_dataset - Copy - coal_prices_10.csv","10md_coal_prices_matched.csv")

#global clean energy index
same_date ("/content/10_year_dataset - Copy - global_clean_index_10.csv","10md_global_clean_energy_index_matched.csv")


In [ ]:
import pandas as pd
import os



def merge_sentiment_with_label(sentiment_path, label_path, output_path):
    # Load datasets
    df_sentiment = pd.read_csv(sentiment_path)
    df_label = pd.read_csv(label_path)

    # Convert to datetime
    df_sentiment['date'] = pd.to_datetime(df_sentiment['date'])
    df_label['date'] = pd.to_datetime(df_label['date'])

    # Sort sentiment values so non-zeros come first
    df_sentiment_sorted = df_sentiment.sort_values(by='sentiment', ascending=False, key=lambda x: x != 0)

    # Keep only one row per day, prioritizing non-zero
    df_sentiment_deduped = df_sentiment_sorted.drop_duplicates(subset='date', keep='first')

    # Merge label with sentiment
    merged = pd.merge(df_label, df_sentiment_deduped, on='date', how='left')
    merged['sentiment'] = merged['sentiment'].fillna(0)
    merged=merged.drop(columns=["label"])

    # Save result
    merged.to_csv(output_path, index=False)

    return merged




merge_sentiment_with_label("/content/10_year_dataset - Copy - sentiment_10.csv","/content/10_year_dataset - Copy - Carbon_Emissions_Futures_10.csv","sentiment_matched.csv")


In [ ]:
# 10 years monthly intrapolated to weekly, weekly frequency for all data


merge_weekly_features_to_labels("/content/features_daily_weekly/10_cpi_weekly.csv", "10cpi_weekly_matched.csv")

merge_weekly_features_to_labels("/content/features_daily_weekly/10_natural_gas_weekly.csv", "10natural_gas_weekly_matched.csv")

merge_weekly_features_to_labels("/content/features_daily_weekly/10_production_industry_weekly.csv", "10production_industry_weekly_matched.csv")

merge_weekly_features_to_labels("/content/features_daily_weekly/euro_stoxx_daily_2_weekly.csv", "10euro_stoxx_weekly_matched.csv")

merge_weekly_features_to_labels("/content/features_daily_weekly/ecb_interest_rate_daily_2_weekly.csv", "10ecb_interest_rate_weekly_matched.csv")

merge_weekly_features_to_labels("/content/features_daily_weekly/crude_daily_2_weekly.csv", "10crude_oil_weekly_matched.csv")

merge_weekly_features_to_labels("/content/features_daily_weekly/coal_daily_2_weekly.csv", "10coal_weekly_matched.csv")

merge_weekly_features_to_labels("/content/features_daily_weekly/global_clean_daily_2_weekly.csv", "10global_clean_weekly_matched.csv")

merge_weekly_features_to_labels("/content/features_daily_weekly/sentiment_daily_2_weekly.csv", "10sentiment_weekly_matched.csv")



In [ ]:

#10md_sentiment

import pandas as pd
import glob
import os

# === Path to your feature CSV files ===
feature_folder = "features/"
feature_files = glob.glob(os.path.join(feature_folder, "*.csv"))

# === Load label dataset ===
df_labels = pd.read_csv("labels_aligned.csv")
df_labels['date'] = pd.to_datetime(df_labels['date'])

# === Start with labels so we can merge into this
df_merged = df_labels.copy()

# === Merge each feature file
for filepath in feature_files:
    df_feature = pd.read_csv(filepath)
    df_feature['date'] = pd.to_datetime(df_feature['date'])

    # Only keep the feature column (excluding Date)
    feature_name = [col for col in df_feature.columns if col != 'date'][0]

    # Merge on Date
    df_merged = df_merged.merge(df_feature, on='date', how='left')

    print(f"? Merged: {feature_name}")

# === Optional: Reorder columns
columns = ['date'] + [col for col in df_merged.columns if col != 'date' and col != 'label'] + ['label']
df_merged = df_merged[columns]

# === Save final dataset
df_merged.to_csv("10md_sentiment.csv", index=False)
print("? Final merged dataset saved to '10d_sentiment.csv'")

df_merged


In [ ]:
#10d_sentiment


import pandas as pd
import glob
import os

# === Path to your feature CSV files ===
feature_folder = "features2/"
feature_files = glob.glob(os.path.join(feature_folder, "*.csv"))

# === Load label dataset ===
df_labels = pd.read_csv("labels_aligned.csv")
df_labels['date'] = pd.to_datetime(df_labels['date'])

# === Start with labels so we can merge into this
df_merged = df_labels.copy()

# === Merge each feature file
for filepath in feature_files:
    df_feature = pd.read_csv(filepath)
    df_feature['date'] = pd.to_datetime(df_feature['date'])

    # Only keep the feature column (excluding Date)
    feature_name = [col for col in df_feature.columns if col != 'date'][0]

    # Merge on Date
    df_merged = df_merged.merge(df_feature, on='date', how='left')

    print(f"? Merged: {feature_name}")

# === Optional: Reorder columns
columns = ['date'] + [col for col in df_merged.columns if col != 'date' and col != 'label'] + ['label']
df_merged = df_merged[columns]

# === Save final dataset
df_merged.to_csv("10d_sentiment.csv", index=False)
print("? Final merged dataset saved to '10d_sentiment.csv'")

df_merged


In [ ]:

#10wm_sentiment

import pandas as pd
import glob
import os

# === Path to your feature CSV files ===
feature_folder = "/content/features_weekly"
feature_files = glob.glob(os.path.join(feature_folder, "*.csv"))

# === Load label dataset ===
df_labels = pd.read_csv("/content/10_year_dataset - Copy - Carbon_emissions_weekly_10.csv")
df_labels['date'] = pd.to_datetime(df_labels['date'])

# === Start with labels so we can merge into this
df_merged = df_labels.copy()

# === Merge each feature file
for filepath in feature_files:
    df_feature = pd.read_csv(filepath)
    df_feature['date'] = pd.to_datetime(df_feature['date'])

    # Only keep the feature column (excluding Date)
    feature_name = [col for col in df_feature.columns if col != 'date'][0]

    # Merge on Date
    df_merged = df_merged.merge(df_feature, on='date', how='left')

    print(f"? Merged: {feature_name}")

# === Optional: Reorder columns
columns = ['date'] + [col for col in df_merged.columns if col != 'date' and col != 'label'] + ['label']
df_merged = df_merged[columns]

# === Save final dataset
df_merged.to_csv("10wm_sentiment.csv", index=False)
print("? Final merged dataset saved to '10d_sentiment.csv'")

df_merged


In [ ]:

#10w_sentiment

import pandas as pd
import glob
import os

# === Path to your feature CSV files ===
feature_folder = "/content/features_weekly_nomonth"
feature_files = glob.glob(os.path.join(feature_folder, "*.csv"))

# === Load label dataset ===
df_labels = pd.read_csv("/content/10_year_dataset - Copy - Carbon_emissions_weekly_10.csv")
df_labels['date'] = pd.to_datetime(df_labels['date'])

# === Start with labels so we can merge into this
df_merged = df_labels.copy()

# === Merge each feature file
for filepath in feature_files:
    df_feature = pd.read_csv(filepath)
    df_feature['date'] = pd.to_datetime(df_feature['date'])

    # Only keep the feature column (excluding Date)
    feature_name = [col for col in df_feature.columns if col != 'date'][0]

    # Merge on Date
    df_merged = df_merged.merge(df_feature, on='date', how='left')

    print(f"? Merged: {feature_name}")

# === Optional: Reorder columns
columns = ['date'] + [col for col in df_merged.columns if col != 'date' and col != 'label'] + ['label']
df_merged = df_merged[columns]

# === Save final dataset
df_merged.to_csv("10w_sentiment.csv", index=False)
print("? Final merged dataset saved to '10d_sentiment.csv'")

df_merged


In [ ]:
# import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt
# import os

# # === Load your merged dataset ===
# df = pd.read_csv("/content/final_dataset.csv")

# # === Optional: Drop Date column for numeric-only analysis
# df_numeric = df.drop(columns=['date'])

# df_numeric= df_numeric.drop(columns=["Deposit facility - date of changes (raw data) - Level (FM.D.U2.EUR.4F.KR.DFR.LEV)", "Marginal lending facility - date of changes (raw data) - Level (FM.D.U2.EUR.4F.KR.MLFR.LEV)", "High", "Open", "Low"])

# # === 1. Correlation Heatmap ===
# plt.figure(figsize=(12, 8))
# corr = df_numeric.corr()
# sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
# plt.title("?? Feature Correlation Heatmap")
# plt.tight_layout()
# #plt.savefig("processed_data/feature_correlation_heatmap.png")
# plt.show()

# # === 2. Pairplot (Optional: only for small number of features) ===
# # This helps visualize distributions and relationships
# if df_numeric.shape[1] <= 7:  # Too many features can clutter this
#     sns.pairplot(df_numeric, hue='label', corner=True)
#     plt.suptitle("?? Feature Pairwise Relationships", y=1.02)
#     #plt.savefig("processed_data/feature_pairplot.png")
#     plt.show()

# # === 3. Target vs Individual Feature (Top 5 correlations with Target) ===
# target_corr = corr['label'].drop('label').abs().sort_values(ascending=False)
# top_features = target_corr.head(5).index.tolist()

# for feature in top_features:
#     plt.figure(figsize=(8, 4))
#     sns.boxplot(x='label', y=feature, data=df)
#     plt.title(f"?? Relationship: {feature} vs label")
#     plt.tight_layout()
#     #plt.savefig(f"processed_data/{feature}_vs_label.png")
#     plt.show()

# print("? All feature relationship plots saved in 'processed_data/'")


In [ ]:
# corr = df_numeric.corr()
# target_corr = corr['label'].drop('label').abs().sort_values(ascending=False)
# top_features = target_corr.head(5).index.tolist()  # Top 5 features

# for feature in top_features:
#     plt.figure(figsize=(8, 5))
#     sns.scatterplot(data=df, x=feature, y='label', alpha=0.6, edgecolor='k')
#     plt.title(f"?? Scatterplot: {feature} vs label")
#     plt.tight_layout()
#     #plt.savefig(f"processed_data/plots/{feature}_vs_target_scatter.png")
#     plt.show()
